# 03 · Evaluación de rigor

Este notebook somete el modelo del notebook **02 · Modelo y métricas** a pruebas que verifican que su rendimiento es real, que no se explica por factores ajenos a la morfología y que se interpreta correctamente:

1. **Comparación de modelos:** el modelo radiómico frente a variantes y líneas base simples (volumen del hipocampo, todos los volúmenes, solo edad y sexo), todos con el mismo protocolo y las mismas particiones.
2. **t-test de remuestreo corregido:** si las diferencias de AUC entre modelos son mayores de lo esperable por azar.
3. **Prueba de permutación:** si cada comparativa supera el azar.
4. **Comparación de algoritmos:** si un clasificador no lineal rinde mejor que la regresión logística.
5. **Horizontes cruzados:** qué detectan realmente los modelos conversor vs Ctrl al aplicarlos a la otra cohorte de conversión.
6. **Análisis longitudinal pareado:** cuánto cambia la morfología en un año dentro del mismo sujeto, frente a la variabilidad entre sujetos.

Las figuras se guardan en `img/` (PNG a 300 dpi). Cuando una celda genera varias figuras del mismo tipo, se guardan sin mostrarse y la celda indica sus rutas.

Todas las evaluaciones usan la validación cruzada repetida y anidada, agrupada por sujeto, descrita en el notebook 02 (`ml_core/evaluation.py`).

## 0. Configuración y carga de datos

Mismos datos, semillas y particiones que el notebook 02. Así el modelo "Radiómica (ajustada)" de este notebook es exactamente el evaluado en el 02, y todos los modelos se comparan fold a fold sobre las mismas particiones.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

# Los procesos de entrenamiento en paralelo (joblib) heredan esta variable y tampoco imprimen advertencias
import os
os.environ['PYTHONWARNINGS'] = 'ignore'

from collections import Counter
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from scipy import stats
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.svm import SVC
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
from ml_core import (GRID_LR, build_pipeline, prepare_comparison, outer_splits, evaluate_nested, summarize,
                     best_threshold, load_region_names, comparativa_label, save_figure, plot_mean_roc,
                     plot_auc_comparison, plot_auc_differences, plot_permutation_null, plot_cross_horizon,
                     plot_longitudinal_pairs, plot_longitudinal_changes, plot_effect_sizes)

DIR_DATOS = './datos_procesados'
# Nombres de las regiones: los de FreeSurfer y su traducción al español (ml_core/regions.py)
NOMBRES_REGION, NOMBRES_REGION_ES = load_region_names('./FreeSurferColorLUT.csv')

N_REPEATS, N_FOLDS = 5, 5
SEED = 0
N_PERMUTATIONS = 1000

COMPARATIVAS = {'1Y_vs_Ctrl': [0, 1], '2Y_vs_Ctrl': [0, 2], '1Y_vs_2Y': [1, 2]}
REFERENCIA_AJUSTE = {'1Y_vs_Ctrl': 0, '2Y_vs_Ctrl': 0, '1Y_vs_2Y': None}
HIPOCAMPO = ['Label_17_original_shape_VoxelVolume', 'Label_53_original_shape_VoxelVolume']
GRID_SIN_SELECCION = {'clasificador__C': [0.01, 0.1, 1.0]}

df = pd.read_csv(os.path.join(DIR_DATOS, 'dataset_procesado.csv'))
demografia = pd.read_csv(os.path.join(DIR_DATOS, 'demografia.csv'))
assert (df['PatientID'] == demografia['PatientID']).all() and (df['Target'] == demografia['Target']).all()
VOXELVOLUME = [c for c in df.columns if c.endswith('original_shape_VoxelVolume')]
print(f"Dataset: {df.shape[0]} imágenes × {df.shape[1] - 3} características de forma")

## 1. Comparación de modelos con el mismo protocolo

Cinco configuraciones, evaluadas con la misma validación anidada y **las mismas particiones**:

| Modelo | Entrada | Para qué sirve |
|---|---|---|
| Radiómica (ajustada) | Las 1498 características de forma, ajustadas por edad y sexo | Modelo principal (el del notebook 02) |
| Radiómica (sin ajuste) | Las mismas, sin ajuste | Muestra cuánto del rendimiento dependía de la diferencia de edad |
| Hipocampo (ajustado) | Solo el VoxelVolume de los hipocampos izquierdo (17) y derecho (53) | Línea base de la volumetría clásica |
| Todos los VoxelVolume (ajustado) | El VoxelVolume de las 107 regiones | Separa el aporte de los descriptores de forma del aporte de los volúmenes |
| Solo edad + sexo | Únicamente las covariables | Rendimiento que se obtiene sin ninguna imagen |

Las líneas base de 2 variables no pasan por el filtro de correlación ni por Lasso; solo se ajusta la `C` de la regresión logística.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

def configuraciones(comparativa):
    """(X, y, groups), pipeline y grid de cada modelo comparado."""
    clases, ref = COMPARATIVAS[comparativa], REFERENCIA_AJUSTE[comparativa]
    return {
        'Radiómica (ajustada)': (prepare_comparison(df, demografia, clases),
                                 build_pipeline(reference_class=ref), GRID_LR),
        'Radiómica (sin ajuste)': (prepare_comparison(df, demografia, clases, covariates=None),
                                   build_pipeline(covariates=None), GRID_LR),
        'Hipocampo (ajustado)': (prepare_comparison(df, demografia, clases, columns=HIPOCAMPO),
                                 build_pipeline(reference_class=ref, selection=False, correlation_filter=False), GRID_SIN_SELECCION),
        'Todos los VoxelVolume (ajustado)': (prepare_comparison(df, demografia, clases, columns=VOXELVOLUME),
                                             build_pipeline(reference_class=ref), GRID_LR),
        'Solo edad + sexo': (prepare_comparison(df, demografia, clases, columns=[]),
                             build_pipeline(covariates=None, selection=False, correlation_filter=False), GRID_SIN_SELECCION),
    }

resultados, datos_modelo, filas = {}, {}, []
for comparativa in COMPARATIVAS:
    for nombre, ((X, y, groups), pipeline, param_grid) in configuraciones(comparativa).items():
        resultados[(comparativa, nombre)] = evaluate_nested(X, y, groups, pipeline, param_grid, N_REPEATS, N_FOLDS, SEED)
        datos_modelo[(comparativa, nombre)] = (X, y, groups, pipeline)
        filas.append({'Comparativa': comparativa, **summarize(resultados[(comparativa, nombre)], nombre)})
        print(f"[{comparativa}] {nombre}: listo")

tabla_modelos = pd.DataFrame(filas).set_index(['Comparativa', 'Modelo'])
tabla_modelos.to_excel('Tabla_Comparacion_Modelos.xlsx')
display(tabla_modelos)

AUC media de los cinco modelos en cada comparativa (la línea negra es ± 1 desviación estándar; la línea punteada, el azar). "Radiómica" es el modelo principal.

In [ ]:
plot_auc_comparison(resultados)

## 2. t-test de remuestreo corregido contra las líneas base

Los dos modelos se evaluaron en las mismas 25 particiones, así que para cada fold *j* hay una diferencia de AUC $d_j$. Un t-test pareado ordinario sobre esas diferencias sería demasiado optimista: los folds comparten la mayor parte de sus datos de entrenamiento, no son independientes y la varianza de $\bar d$ queda subestimada.

La corrección de Nadeau y Bengio (2003) agranda la varianza según la proporción entre test y train:

$$t = \frac{\bar d}{\sqrt{\left(\frac{1}{k} + \frac{n_{test}}{n_{train}}\right) s_d^2}}, \qquad gl = k - 1$$

donde $k = 25$ es el número de folds y $s_d^2$ la varianza de las diferencias. **p** es la probabilidad de observar una diferencia al menos tan grande si los dos modelos tuvieran el mismo AUC.

In [ ]:
def corrected_ttest(results_a, results_b, metric='AUC'):
    """t-test de remuestreo corregido (Nadeau y Bengio, 2003). Devuelve (diferencia media, t, p bilateral)."""
    a, b = results_a['folds'], results_b['folds']
    assert a[['Repeticion', 'Fold', 'n_test']].equals(b[['Repeticion', 'Fold', 'n_test']]), "Particiones distintas"
    d = (a[metric] - b[metric]).to_numpy()
    k = len(d)
    test_train_ratio = a['n_test'].mean() / a['n_train'].mean()
    t = d.mean() / np.sqrt((1 / k + test_train_ratio) * d.var(ddof=1))
    return d.mean(), t, 2 * stats.t.sf(abs(t), df=k - 1)

MODELOS_COMPARADOS = ['Radiómica (sin ajuste)', 'Hipocampo (ajustado)', 'Todos los VoxelVolume (ajustado)', 'Solo edad + sexo']

filas, diferencias_por_fold, valores_p = [], {}, {}
for comparativa in COMPARATIVAS:
    principal = resultados[(comparativa, 'Radiómica (ajustada)')]
    for base in MODELOS_COMPARADOS:
        diferencia, t, p = corrected_ttest(principal, resultados[(comparativa, base)])
        diferencias_por_fold[(comparativa, base)] = (principal['folds']['AUC'] - resultados[(comparativa, base)]['folds']['AUC']).to_numpy()
        valores_p[(comparativa, base)] = p
        filas.append({'Comparativa': comparativa, 'Comparado con': base,
                      'Diferencia de AUC (radiómica − comparado)': round(diferencia, 3), 't': round(t, 2), 'p': p})
tabla_lineas_base = pd.DataFrame(filas).set_index(['Comparativa', 'Comparado con'])
tabla_lineas_base.to_excel('Tabla_Comparacion_Lineas_Base.xlsx')
display(tabla_lineas_base)

In [ ]:
# Diferencia media de AUC frente a cada modelo comparado; el asterisco marca las diferencias con p < 0.05
plot_auc_differences(diferencias_por_fold, valores_p)

### 2.1 ¿El modelo sin ajuste usa la edad?

El modelo sin ajuste por edad y sexo obtiene un AUC algo mayor que el ajustado, aunque la diferencia no es significativa. Como los conversores son en promedio unos cuatro años mayores que los Ctrl, esa ventaja podría venir de la edad y no de la enfermedad.

Para comprobarlo se usan solo los **Ctrl**, que nunca convirtieron, con la probabilidad fuera de fold de cada modelo (promedio de las 5 repeticiones):
- **Correlación de Spearman entre la edad y la probabilidad de conversión.** En un modelo que no usara la edad debería ser cercana a 0.
- **Porcentaje de Ctrl clasificados como conversores**, con el umbral de su fold, en los más jóvenes y los mayores (partidos por la mediana de edad de los Ctrl).

Si el modelo sin ajuste usa la edad, marcará como conversores sobre todo a los Ctrl mayores.

In [ ]:
filas = []
for comparativa in ['1Y_vs_Ctrl', '2Y_vs_Ctrl']:
    edad = demografia.loc[df['Target'].isin(COMPARATIVAS[comparativa]), 'Edad'].to_numpy()
    for nombre in ['Radiómica (ajustada)', 'Radiómica (sin ajuste)']:
        res = resultados[(comparativa, nombre)]
        ctrl = res['y'] == 0
        mayores = edad >= np.median(edad[ctrl])
        # Clasificación de cada imagen cuando estuvo en la partición de prueba, con el umbral de su fold
        clasificado = np.zeros(res['probs_oof'].shape)
        for (r, _, _, idx_test), threshold in zip(res['splits'], res['folds']['Umbral']):
            clasificado[r, idx_test] = res['probs_oof'][r, idx_test] >= threshold
        filas.append({'Comparativa': comparativa, 'Modelo': nombre,
                      'Spearman edad–probabilidad (Ctrl)': stats.spearmanr(edad[ctrl], res['probs_oof'].mean(axis=0)[ctrl]).statistic,
                      'Ctrl jóvenes clasificados como conversor (%)': 100 * clasificado[:, ctrl & ~mayores].mean(),
                      'Ctrl mayores clasificados como conversor (%)': 100 * clasificado[:, ctrl & mayores].mean()})

tabla_edad = pd.DataFrame(filas).set_index(['Comparativa', 'Modelo'])
tabla_edad.to_excel('Tabla_Efecto_Edad.xlsx')
display(tabla_edad.round(2))

Curvas ROC medias del modelo radiómico frente a la línea base del hipocampo y a la de solo edad y sexo.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5.2))
for ax, comparativa in zip(axes, COMPARATIVAS):
    curvas = {nombre: resultados[(comparativa, nombre)] for nombre in ['Radiómica (ajustada)', 'Hipocampo (ajustado)', 'Solo edad + sexo']}
    plot_mean_roc(curvas, title=comparativa_label(comparativa), ax=ax)
save_figure("ROC_Lineas_Base")

## 3. Prueba de permutación

Mide si el AUC supera lo que el mismo pipeline obtendría por azar con estos mismos datos (Ojala y Garriga, 2010):

1. **AUC observado:** validación cruzada de 5 folds, agrupada por sujeto, con las etiquetas reales.
2. **Distribución nula:** se barajan las etiquetas 1000 veces y se repite el pipeline completo con cada barajado. Las imágenes no cambian y la proporción de clases tampoco; solo se rompe la relación entre morfología y diagnóstico.
3. **p** = (1 + número de permutaciones con AUC ≥ observado) / (1 + 1000). El mínimo posible es 1/1001 ≈ 0.001, que se reporta como p < 0.001.

Repetir la validación anidada 1000 veces sería inviable. Por eso se fijan los hiperparámetros elegidos con más frecuencia en la sección 1, y el mismo pipeline fijo se aplica a los datos reales y a los permutados. El AUC observado de esta prueba sale un poco más alto que el de la validación anidada; la cifra de rendimiento que se reporta es la anidada.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

def modal_params(results):
    """Hiperparámetros elegidos con más frecuencia en los folds externos."""
    conteo = Counter(tuple(sorted(p.items())) for p in results['folds']['Hiperparametros'])
    return dict(conteo.most_common(1)[0][0])

def auc_cv(pipeline, X, y, groups, cv):
    aucs = []
    for idx_train, idx_test in cv.split(X, y, groups):
        model = clone(pipeline).fit(X.iloc[idx_train], y[idx_train])
        aucs.append(roc_auc_score(y[idx_test], model.predict_proba(X.iloc[idx_test])[:, 1]))
    return np.mean(aucs)

def permutation_test(pipeline, X, y, groups, n_permutations, n_folds=5, seed=0):
    """AUC observado, AUCs con etiquetas permutadas y p de la prueba de permutación."""
    cv = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    observed = auc_cv(pipeline, X, y, groups, cv)
    rng = np.random.default_rng(seed)
    permuted_labels = [rng.permutation(y) for _ in range(n_permutations)]
    null_aucs = np.array(Parallel(n_jobs=-1)(delayed(auc_cv)(pipeline, X, y_perm, groups, cv) for y_perm in permuted_labels))
    p_value = (1 + np.sum(null_aucs >= observed)) / (1 + n_permutations)
    return observed, null_aucs, p_value

filas, distribuciones_nulas = [], {}
for comparativa in COMPARATIVAS:
    X, y, groups, pipeline = datos_modelo[(comparativa, 'Radiómica (ajustada)')]
    params = modal_params(resultados[(comparativa, 'Radiómica (ajustada)')])
    observed, null_aucs, p_value = permutation_test(clone(pipeline).set_params(**params), X, y, groups, N_PERMUTATIONS, N_FOLDS, SEED)
    filas.append({'Comparativa': comparativa, 'Hiperparámetros fijos': params, 'AUC observado': round(observed, 3),
                  'AUC nulo (media)': round(null_aucs.mean(), 3), 'AUC nulo (percentil 95)': round(np.quantile(null_aucs, 0.95), 3),
                  'p': p_value})
    distribuciones_nulas[comparativa] = (null_aucs, observed, p_value)

tabla_permutacion = pd.DataFrame(filas).set_index('Comparativa')
tabla_permutacion.to_excel('Tabla_Prueba_Permutacion.xlsx')
display(tabla_permutacion)

Histograma del AUC con etiquetas permutadas (gris) y AUC observado con las etiquetas reales (línea roja), con su valor p. Si la línea queda fuera del histograma, el modelo supera al azar. Se guarda además una figura por comparativa.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
rutas = []
for ax, (comparativa, (null_aucs, observed, p_value)) in zip(axes, distribuciones_nulas.items()):
    plot_permutation_null(null_aucs, observed, p_value, comparativa, ax=ax)
    # La misma gráfica como figura individual: se guarda sin mostrarse
    rutas.append(plot_permutation_null(null_aucs, observed, p_value, comparativa, show=False))
plt.figure(fig.number)
save_figure("Permutacion")
print("Figuras individuales guardadas:")
print('\n'.join(rutas))

## 4. Comparación de algoritmos

Se cambia solo el clasificador final; el ajuste, los filtros y la selección Lasso son los mismos. Los grids son reducidos:

| Algoritmo | Hiperparámetros explorados |
|---|---|
| Regresión logística | C de Lasso, C de la regresión (grid del modelo principal) |
| SVM | C de Lasso; C y kernel (lineal, RBF) |
| Random Forest (300 árboles) | C de Lasso; profundidad máxima, mínimo de muestras por hoja |
| XGBoost (300 árboles) | C de Lasso; profundidad máxima |

Todos compensan el desbalance de clases: `class_weight='balanced'` en scikit-learn y `scale_pos_weight` en XGBoost. Si un modelo no lineal no supera claramente a la regresión logística, se prefiere esta por ser interpretable.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

def algoritmos(y):
    """{nombre: (clasificador, grid)}; None usa la regresión logística del pipeline principal."""
    grid_l1 = {'seleccion__C': [0.05, 0.1, 0.2]}
    return {
        'Regresion_Logistica': (None, GRID_LR),
        'SVM': (SVC(probability=True, class_weight='balanced', random_state=42),
                {**grid_l1, 'clasificador__C': [0.1, 1.0, 10.0], 'clasificador__kernel': ['linear', 'rbf']}),
        'Random_Forest': (RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42, n_jobs=1),
                          {**grid_l1, 'clasificador__max_depth': [None, 5], 'clasificador__min_samples_leaf': [1, 3]}),
        'XGBoost': (XGBClassifier(n_estimators=300, learning_rate=0.05, subsample=0.8, eval_metric='logloss',
                                  scale_pos_weight=(y == 0).sum() / (y == 1).sum(), random_state=42, n_jobs=1),
                    {**grid_l1, 'clasificador__max_depth': [2, 3]}),
    }

filas, resultados_algoritmos = [], {}
for comparativa in ['1Y_vs_Ctrl', '2Y_vs_Ctrl']:
    X, y, groups, _ = datos_modelo[(comparativa, 'Radiómica (ajustada)')]
    for nombre, (classifier, param_grid) in algoritmos(y).items():
        pipeline = build_pipeline(reference_class=REFERENCIA_AJUSTE[comparativa], classifier=classifier)
        results = evaluate_nested(X, y, groups, pipeline, param_grid, N_REPEATS, N_FOLDS, SEED)
        resultados_algoritmos[(comparativa, nombre)] = results
        filas.append({'Comparativa': comparativa, **summarize(results, nombre)})
        print(f"[{comparativa}] {nombre}: listo")

tabla_algoritmos = pd.DataFrame(filas).set_index(['Comparativa', 'Modelo'])
tabla_algoritmos.to_excel('Tabla_Comparacion_Algoritmos.xlsx')
display(tabla_algoritmos)

AUC media de cada algoritmo en las dos comparativas con modelo (la línea negra es ± 1 desviación estándar).

In [ ]:
plot_auc_comparison(resultados_algoritmos, filename="Comparacion_Algoritmos")

## 5. Horizontes cruzados

El modelo 1Y_vs_Ctrl se aplica a las imágenes 2Y, y el modelo 2Y_vs_Ctrl a las imágenes 1Y. Siempre fuera de fold: en cada fold, solo se predicen las imágenes de la otra cohorte cuyo sujeto no está en el train.

- Si un modelo distinguiera el **horizonte** (12 frente a 24 meses antes del diagnóstico), clasificaría la otra cohorte de conversión como Ctrl.
- Si detecta la **conversión** en general, la clasificará como conversora en una proporción parecida a la de su propia cohorte.

Se usan los hiperparámetros más frecuentes de la sección 1, y el umbral de cada fold se elige en la validación interna de su train.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

def horizontes_cruzados(clases_modelo, clase_externa, pipeline, n_repeats, seed):
    """Probabilidad media y % clasificado como conversor, por cohorte, de un modelo conversor vs Ctrl."""
    X, y, groups = prepare_comparison(df, demografia, clases_modelo)
    X_ext, _, groups_ext = prepare_comparison(df, demografia, [clase_externa])
    probs_ext = [[] for _ in range(len(X_ext))]
    preds_ext = [[] for _ in range(len(X_ext))]
    probs_int, preds_int = np.zeros((n_repeats, len(y))), np.zeros((n_repeats, len(y)))

    for r, f, idx_train, idx_test in outer_splits(y, groups, n_repeats, N_FOLDS, seed):
        model = clone(pipeline).fit(X.iloc[idx_train], y[idx_train])
        inner_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed + 100 * r + f)
        probs_inner = cross_val_predict(clone(pipeline), X.iloc[idx_train], y[idx_train], groups=groups[idx_train],
                                        cv=inner_cv, method='predict_proba')[:, 1]
        threshold = best_threshold(y[idx_train], probs_inner)
        probs = model.predict_proba(X.iloc[idx_test])[:, 1]
        probs_int[r, idx_test], preds_int[r, idx_test] = probs, probs >= threshold
        # Imágenes de la otra cohorte cuyo sujeto no participó en el entrenamiento de este fold
        no_vistas = np.flatnonzero(~np.isin(groups_ext, groups[idx_train]))
        for i, p in zip(no_vistas, model.predict_proba(X_ext.iloc[no_vistas])[:, 1]):
            probs_ext[i].append(p)
            preds_ext[i].append(p >= threshold)

    nombres = {0: 'Ctrl', 1: '1Y', 2: '2Y'}
    filas = [{'Cohorte': nombres[clase], 'n': int((y == etiqueta).sum()),
              'Probabilidad_Media': probs_int[:, y == etiqueta].mean(),
              'Clasificados_Conversor_%': 100 * preds_int[:, y == etiqueta].mean()}
             for etiqueta, clase in [(0, min(clases_modelo)), (1, max(clases_modelo))]]
    filas.append({'Cohorte': f"{nombres[clase_externa]} (no vista en entrenamiento)", 'n': len(X_ext),
                  'Probabilidad_Media': np.mean([np.mean(p) for p in probs_ext]),
                  'Clasificados_Conversor_%': 100 * np.mean([np.mean(p) for p in preds_ext])})
    return pd.DataFrame(filas).set_index('Cohorte')

tablas = {}
for comparativa, clase_externa in [('1Y_vs_Ctrl', 2), ('2Y_vs_Ctrl', 1)]:
    _, _, _, pipeline = datos_modelo[(comparativa, 'Radiómica (ajustada)')]
    params = modal_params(resultados[(comparativa, 'Radiómica (ajustada)')])
    tablas[comparativa] = horizontes_cruzados(COMPARATIVAS[comparativa], clase_externa,
                                              clone(pipeline).set_params(**params), N_REPEATS, SEED)
tabla_horizontes = pd.concat(tablas, names=['Modelo'])
tabla_horizontes.to_excel('Tabla_Horizontes_Cruzados.xlsx')
display(tabla_horizontes.round(3))

Porcentaje de imágenes de cada cohorte que cada modelo clasifica como conversoras. La barra rayada es la cohorte de conversión que el modelo nunca usó en su entrenamiento.

In [ ]:
plot_cross_horizon(tablas)

## 6. Análisis longitudinal pareado

91 sujetos tienen una imagen 2Y y otra 1Y, adquiridas con un año de diferencia. Para cada región se compara el **VoxelVolume crudo** de ambas imágenes del mismo sujeto. Al ser el mismo cerebro, el tamaño de la cabeza se cancela y no hace falta normalizar. Por región se calcula:

- **Cambio mediano (%)** entre la imagen 2Y y la 1Y.
- **Wilcoxon de rangos con signo:** prueba pareada no paramétrica de que el cambio es distinto de cero.
- **p FDR:** p corregido por Benjamini-Hochberg al probar 107 regiones; controla la proporción esperada de falsos positivos.
- **d intra-sujeto** = cambio medio / DE del cambio. Es el efecto que vería un estudio longitudinal.
- **d entre sujetos** = (media 1Y − media 2Y) / DE combinada, sobre todas las imágenes de ambas cohortes. Es el efecto que ve un clasificador transversal, como el de 1Y_vs_2Y.

Si el cambio intra-sujeto es claro pero la diferencia entre cohortes es pequeña frente a la variabilidad entre personas, eso explica por qué 1Y_vs_2Y no supera el azar.

In [ ]:
forma_cruda = pd.read_csv(os.path.join(DIR_DATOS, 'forma_cruda.csv'))

imagenes_2y = forma_cruda[forma_cruda['Target'] == 2].set_index('Sujeto_Real')
imagenes_1y = forma_cruda[forma_cruda['Target'] == 1].set_index('Sujeto_Real')
sujetos_pareados = imagenes_2y.index.intersection(imagenes_1y.index).sort_values()
antes, despues = imagenes_2y.loc[sujetos_pareados], imagenes_1y.loc[sujetos_pareados]

filas = []
for columna in [c for c in forma_cruda.columns if c.endswith('original_shape_VoxelVolume')]:
    v_antes, v_despues = antes[columna].to_numpy(float), despues[columna].to_numpy(float)
    cambio = v_despues - v_antes
    de_combinada = np.sqrt((imagenes_1y[columna].var() + imagenes_2y[columna].var()) / 2)
    label = int(columna.split('_')[1])
    filas.append({'LabelID': label, 'Region': NOMBRES_REGION_ES.get(label, ''), 'Region_FreeSurfer': NOMBRES_REGION.get(label, ''),
                  'Cambio_Mediano_%': np.median(100 * cambio / v_antes),
                  'p_Wilcoxon': stats.wilcoxon(v_despues, v_antes).pvalue,
                  'd_intra': cambio.mean() / cambio.std(ddof=1),
                  'd_entre': (imagenes_1y[columna].mean() - imagenes_2y[columna].mean()) / de_combinada})

tabla_longitudinal = pd.DataFrame(filas)
tabla_longitudinal['p_FDR'] = stats.false_discovery_control(tabla_longitudinal['p_Wilcoxon'])
tabla_longitudinal = tabla_longitudinal.sort_values('p_Wilcoxon').reset_index(drop=True)
tabla_longitudinal.to_excel('Tabla_Longitudinal_Pareada.xlsx', index=False)

print(f"{len(sujetos_pareados)} pares; {(tabla_longitudinal['p_FDR'] < 0.05).sum()} de {len(tabla_longitudinal)} "
      f"regiones con cambio significativo (FDR < 0.05)")
print(f"|d| mediana en esas regiones: intra-sujeto {tabla_longitudinal.loc[tabla_longitudinal['p_FDR'] < 0.05, 'd_intra'].abs().median():.2f}, "
      f"entre sujetos {tabla_longitudinal.loc[tabla_longitudinal['p_FDR'] < 0.05, 'd_entre'].abs().median():.2f}")
redondeo = {'Cambio_Mediano_%': 2, 'd_intra': 3, 'd_entre': 3}
display(tabla_longitudinal.drop(columns='Region_FreeSurfer').head(15).round(redondeo))

Volumen mediano de los hipocampos de los 91 sujetos en su imagen 2Y y, un año después, en su imagen 1Y.

In [ ]:
plot_longitudinal_pairs(antes, despues, {'Label_17_original_shape_VoxelVolume': NOMBRES_REGION_ES[17],
                                         'Label_53_original_shape_VoxelVolume': NOMBRES_REGION_ES[53]})

Cambio mediano en un año de las 15 regiones con el cambio más significativo (las de menor valor p, ordenadas por su cambio): los ventrículos se expanden y las regiones temporales y los hipocampos se atrofian.

In [ ]:
plot_longitudinal_changes(tabla_longitudinal, NOMBRES_REGION_ES, top=15)

Tamaño del efecto de un año en las 10 regiones con el cambio más significativo, medido de dos formas:
- **Azul:** dentro del sujeto (su imagen 2Y frente a su imagen 1Y).
- **Gris:** entre las cohortes 1Y y 2Y, es decir, comparando personas distintas.

El cambio de un año es claro al seguir a la misma persona, pero casi invisible al comparar personas distintas.

In [ ]:
plot_effect_sizes(tabla_longitudinal, NOMBRES_REGION_ES, top=10)